# Task 3 Pilot: 30-Step SFT GPU Pilot (`align-sft-pilot`)

In [ ]:
!nvidia-smi
import os, shutil, pathlib

input_root = pathlib.Path('/kaggle/input')
src_dir = next(input_root.rglob('submission_adapter.py')).parent
work_repo = pathlib.Path('/tmp/repo')
if work_repo.exists():
    shutil.rmtree(work_repo)
shutil.copytree(src_dir, work_repo)
os.chdir(work_repo)
print('Found code dataset at:', src_dir)
print('COMMIT:', (work_repo / 'COMMIT.txt').read_text().strip())


In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors
!python -m pip install -q -e . --no-deps
import torch, transformers, datasets
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available(), 'GPUs:', torch.cuda.device_count())
print('Transformers:', transformers.__version__, 'Datasets:', datasets.__version__)


In [ ]:
# Run 30 optimizer steps on 2,000 train examples, validate on 200 examples
!PYTHONPATH=. python -m src.align.train_sft --config configs/sft_pilot.yaml


In [ ]:
# Sanity-check generation on 3 prompts from the saved pilot checkpoint
!PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path /kaggle/working/outputs/sft_pilot/best \
  --eval_set_path eval_sets/alpaca_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/sft_pilot/pilot_sanity_3.json \
  --generator_name qwen2.5-0.5b-sft-pilot \
  --dataset_name alpaca_eval \
  --limit 3 \
  --batch_size 3

# Remove checkpoint weights from pilot run so output download stays tiny
import shutil
shutil.rmtree('/kaggle/working/outputs/sft_pilot/best', ignore_errors=True)
shutil.rmtree('/kaggle/working/outputs/sft_pilot/final', ignore_errors=True)
